# Swahili ASR & NLP Engineering Pipeline

**Practical Assessment — Final Clean Notebook**

This notebook documents the completed experiment and provides a reproducible path to inspect or rerun the pipeline. Expensive operations are disabled by default to avoid repeatedly downloading/decoding the full FLEURS dataset in constrained Colab runtimes.


In [ ]:
from pathlib import Path
import json
import platform
import torch

PROJECT_DIR = Path("/content/drive/MyDrive/Swahili_ASR_NLP_Assessment")
RUN_EXPENSIVE_EXPERIMENTS = False

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


## 1. Data Engineering

The experiments use Google FLEURS Swahili (`sw_ke`). The evaluated dataset contained 3,070 training samples, 211 validation samples, and 487 test samples. Total evaluated duration was approximately 15.4 hours.

Audio is decoded on demand. This avoids keeping every waveform in RAM and is important for free-tier Colab runtimes.


In [ ]:
from datasets import load_dataset

if RUN_EXPENSIVE_EXPERIMENTS:
    fleurs = load_dataset("google/fleurs", "sw_ke", trust_remote_code=False)
    print(fleurs)
else:
    print("Dataset loading disabled by default.")
    print("Set RUN_EXPENSIVE_EXPERIMENTS=True to intentionally rerun the experiment.")


In [ ]:
import re
import unicodedata

def normalize_swahili_text(text):
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

print(normalize_swahili_text(
    "Akawatumia wanafunzi wa chuo kikuu chaa ndani barua pepe."
))


## 2. ASR Baseline and Fine-Tuning

**Base model:** `openai/whisper-small`

A pilot fine-tuning run used 1,000 training samples and 100 validation samples for one epoch on a Tesla T4. The resulting checkpoint was `checkpoint-125`.

The full 487-sample test evaluation was completed and saved before the Colab runtime became compute-limited.


In [ ]:
ASR_RESULTS = PROJECT_DIR / "results" / "final_asr_results.json"

if ASR_RESULTS.exists():
    asr = json.loads(ASR_RESULTS.read_text(encoding="utf-8"))
    print("ASR test samples:", asr["test_samples"])
    print("Baseline WER:", asr["baseline"]["wer"])
    print("Fine-tuned WER:", asr["fine_tuned"]["wer"])
    print("Baseline CER:", asr["baseline"]["cer"])
    print("Fine-tuned CER:", asr["fine_tuned"]["cer"])
    print("WER improvement:", asr["improvement_percent"]["wer"], "%")
    print("CER improvement:", asr["improvement_percent"]["cer"], "%")
else:
    print("ASR results file not found.")


## 3. Reasoning Fine-Tuning

**Base model:** `Qwen/Qwen2.5-1.5B-Instruct`

QLoRA v1 used full-sequence language-model loss and achieved 0% Exact Match on the held-out 100-example benchmark.

QLoRA v2 changed the objective to **assistant-only loss** and used a concise Swahili instruction prompt. This produced 97/100 correct answers.


In [ ]:
REASONING_RESULTS = PROJECT_DIR / "results" / "final_reasoning_results.json"

if REASONING_RESULTS.exists():
    reasoning = json.loads(REASONING_RESULTS.read_text(encoding="utf-8"))
    print("Reasoning test samples:", reasoning["test_samples"])
    print("Baseline EM:", reasoning["baseline"]["exact_match"])
    print("QLoRA v1 EM:", reasoning["qlora_v1"]["exact_match"])
    print("QLoRA v2 EM:", reasoning["qlora_v2"]["exact_match"])
    print("QLoRA v2 correct:", reasoning["qlora_v2"]["correct"], "/ 100")
    print("QLoRA v2 errors:", reasoning["qlora_v2"]["incorrect"])
else:
    print("Reasoning results file not found.")


## 4. End-to-End Architecture

```text
Audio input
   ↓
16 kHz audio preprocessing
   ↓
Whisper Small — Swahili ASR
   ↓
Swahili text normalization
   ↓
Qwen2.5-1.5B-Instruct + QLoRA
   ↓
Short textual reasoning response
```

The saved demonstration audio is an 18-second mono 16 kHz FLEURS sample. The original end-to-end inference was designed to use the fine-tuned Whisper checkpoint and QLoRA v2 adapter.


In [ ]:
DEMO_AUDIO = (
    PROJECT_DIR / "data" / "processed" / "end_to_end_demo"
    / "fleurs_sw_demo_1999.wav"
)
WHISPER_CHECKPOINT = (
    PROJECT_DIR / "checkpoints" / "whisper-small-swahili" / "checkpoint-125"
)
QLORA_CHECKPOINT = (
    PROJECT_DIR / "models" / "qwen2.5-1.5b-qlora-v2" / "checkpoint-100"
)

for name, path in {
    "Demo audio": DEMO_AUDIO,
    "Whisper checkpoint": WHISPER_CHECKPOINT,
    "QLoRA v2 checkpoint": QLORA_CHECKPOINT,
}.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")


## 5. Task 4 — Edge Deployment & Optimization

Task 4 is implemented with a component-specific compression strategy and a reproducible benchmark.

**Reasoning:** Qwen2.5-1.5B-Instruct uses 4-bit NF4 quantization with double quantization and FP16 compute during QLoRA training/inference.

**ASR:** Whisper Small is retained in FP16 for the measured benchmark because its T4 footprint is already modest and no separate quantized-ASR WER/CER experiment was completed.

**Local packaging:** the Whisper fine-tuned checkpoint and QLoRA adapter are stored as separate artifacts; large weights are intentionally excluded from Git.

The benchmark below records the actual Tesla T4 measurement rather than claiming embedded-device performance.

In [ ]:
EDGE_BENCHMARK = PROJECT_DIR / "results" / "edge_benchmark.json"
if EDGE_BENCHMARK.exists():
    edge = json.loads(EDGE_BENCHMARK.read_text(encoding="utf-8"))
    print(json.dumps(edge, indent=2, ensure_ascii=False))
else:
    print("Edge benchmark artifact not found.")

print("\nReproducible benchmark utility: src/deployment/benchmark.py")
print("Quantization helper: src/deployment/quantization.py")
print("Local-serving manifest: src/deployment/serve.py")

### Optional rerun: benchmark the deployed checkpoint

The following is intentionally not executed automatically. It can be run when a GPU is available and should be used for target-device validation. It does not alter the saved 487-sample ASR evaluation.

In [ ]:
# Example:
# from src.deployment.benchmark import benchmark_callable
# results = benchmark_callable(transcribe_fn, demo_inputs, warmup=1)
# print(results)
#
# For a quantized ASR experiment, compare WER/CER against the saved FP16
# result before accepting the compressed model for deployment.

## 6. Final Results

See `results/final_asr_results.json`, `results/final_reasoning_results.json`, and `results/edge_benchmark.json` for the consolidated quantitative artifacts.

### ASR
- Baseline WER: **1.4804**
- Fine-tuned WER: **0.4298**
- WER improvement: **70.97%**
- Baseline CER: **0.7637**
- Fine-tuned CER: **0.1224**
- CER improvement: **83.97%**

### Reasoning
- Baseline Exact Match: **0.00**
- QLoRA V2 Exact Match: **0.97 (97/100)**

### Edge
- Whisper Small: **1.113 s/sample average latency**
- Peak GPU memory: **3.14 GB**
- Hardware: **Tesla T4, 14.56 GB VRAM**
- Reasoning quantization: **4-bit NF4 + double quantization**

The assessment deliverables are therefore: modular code, executable notebook, consolidated README, technical report, quantitative result JSON files, and an explicit edge deployment/optimization strategy.